<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Generate final post-processing figures from scenario run outputs.
- **Primary inputs**: runs/<run_name>/*/output.csv and optional runs/<run_name>/policies/indicator.csv
- **Primary outputs**: publication/report figures
- **Refactor role**: Keep as reporting notebook; replace duplicated output.csv loading block with shared helper.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load scenario outputs for a selected reporting run.
2. Call plotting utilities from `project.write_output` for standardized figures.
3. Optionally include policy indicator overlays for selected scenarios.

### Inputs
- `runs/<run_name>/*/output.csv`
- `runs/<run_name>/policies/indicator.csv` (if available)

### Outputs
- Comparison figures in the selected run output folders.

### How To Reuse
- Set the `name` variable to a run folder available under `reporting/runs/`.


In [ ]:
import pandas as pd
import os
from copy import deepcopy
import sys

# Add the project directory to the path
sys.path.append(os.path.join(os.getcwd(), '..', '..', '..', '..'))

from project.write_output import plot_compare_scenarios, indicator_policies, make_summary
from project.utils import get_json, get_pandas, get_series

In [ ]:
# read results
name = 'ban'
# Refactor: reporting run data is under reporting/runs
folder = os.path.join('runs', name)

# in folder open all child folders and read the output.csv files
dict_output = {child_folder: pd.read_csv(os.path.join(folder, child_folder, 'output.csv'), header=[0], index_col=[0])
             for child_folder in os.listdir(folder)
             if os.path.isdir(os.path.join(folder, child_folder))
             and os.path.isfile(os.path.join(folder, child_folder, 'output.csv'))}

# convert all columns to int
for scenario in dict_output.keys():
    dict_output[scenario].columns = [int(c) for c in dict_output[scenario].columns]

reference, order_scenarios, colors_scenarios = 'Reference', None, None

In [ ]:
if True:
    for scenario in dict_output.keys():
        dict_output[scenario][2050] = dict_output[scenario][2049]


if False:
    scenario_rename = {
    'Reference': 'No policy',
    'BaselinePackage': 'Baseline Package',
    'BaselinePackage + Ban': 'Baseline Package + Ban',
    'SCCBenchmark': 'SCC Benchmark',
    'NoFriction': 'No friction'
    }
    
    dict_output = {scenario_rename.get(k, k): v for k, v in dict_output.items()}
    
    order_scenarios = ['No policy', 'Baseline Package', 'Baseline Package + Ban', 'SCC Benchmark', 'No friction']
    order_scenarios = [i for i in order_scenarios if i in dict_output.keys()]#, 'Regulation'] #
    #order_scenarios = ['2024 Package', '2024 Package + Ban']
    
    colors_scenarios = {
        'No policy': '#577590',
        'Baseline Package': '#f8961e',
        'Baseline Package + Ban': '#f94144',
        'SCC Benchmark': '#43aa8b',
        'No friction': '#f3722c'
    }
    
    
    dict_output = {i:item for i, item in dict_output.items() if i in order_scenarios}
    
    reference = 'No policy'


if False:
    scenario_rename = {
        'NoPolicy': 'No policy',
        'CurrentPolicies': '2018 Package',
        'Reference': '2021 Package',
        'Package2024': '2024 Package',
        'Package2024Zil': '2024 Package + Zil',
        'Package2024Ban': '2024 Package + Ban',
        'CarbonTax': 'Carbon tax',
        'Regulation': 'Regulation',
        'NoMF': 'No friction'
    }
    
    dict_output = {scenario_rename.get(k, k): v for k, v in dict_output.items()}
    
    order_scenarios = ['No policy', '2018 Package', '2021 Package', '2024 Package',  '2024 Package + Ban', 'Carbon tax', 'No friction']
    order_scenarios = [i for i in order_scenarios if i in dict_output.keys()]#, 'Regulation'] #
    #order_scenarios = ['2024 Package', '2024 Package + Ban']
    
    colors_scenarios = {
        'No policy': '#577590',
        '2018 Package': '#90be6d',
        '2021 Package': '#f9c74f',
        '2024 Package': '#f8961e',
        'Without ban': '#f8961e',
        '2024 Package + Ban': '#f94144',
        'With ban': '#f94144',
        'Carbon tax': '#43aa8b',
        'Regulation': '#f3722c', 
        'No friction': '#577590'
    }
    
    
    dict_output = {i:item for i, item in dict_output.items() if i in order_scenarios}
    
    reference = 'No policy' #'2021 Package'
     
if False:
    scenario_rename = {
        'NoPolicyHeater': 'No policy heater',
        'Ban': '2024 Package + Ban',
        'BanNoPolicyHeater': 'Ban',
        'NoPolicyHeaterTest': 'No policy heater test',
        'PolicyHeaterFirstBest': 'First best',
        'Reference': '2024 Package',
        'CarbonTax': 'Carbon tax',
        'NoMF': 'No friction'
    }
    
    dict_output = {scenario_rename.get(k, k): v for k, v in dict_output.items()}
    
    order_scenarios = ['No policy heater', '2024 Package', 'Ban',  '2024 Package + Ban', 'First best', 'Carbon tax', 'No friction', 'No policy heater test']
    order_scenarios = [i for i in order_scenarios if i in dict_output.keys()]#, 'Regulation'] #
    #order_scenarios = ['2024 Package', '2024 Package + Ban']
    
    colors_scenarios = {
        'No policy': '#577590',
        '2018 Package': '#90be6d',
        '2021 Package': '#f9c74f',
        '2024 Package': '#f8961e',
        'Without ban': '#f8961e',
        '2024 Package + Ban': '#f94144',
        'With ban': '#f94144',
        'Carbon tax': '#43aa8b',
        'Regulation': '#f3722c', 
        'No friction': '#577590',
        'First best': '#90be6d',
        'Ban': '#f94144',
        'No policy heater test': '#f3722c',
        'Ban test': '#f3722c',
        'No policy heater': '#f8961e',
    }
    
    
    dict_output = {i:item for i, item in dict_output.items() if i in order_scenarios}
    
    reference = 'No policy heater' #'2021 Package'
    
    
    if False:
        for scenario in dict_output.keys():
            dict_output[scenario][2050] = dict_output[scenario][2049]
            
if False:
    scenario_rename = {'carbon_tax' : 'Carbon tax',
                    'carbon_tax_high' : 'Carbon tax + EU-ETS 2',
                    'cee_2018' : 'WCO 2018',
                    'cee_2021' : 'WCO 2021',
                    'cee_2024' : 'WCO 2024',
                    'obligation' : 'Mandatory renovation',
                    'restriction_gas' : 'Ban gas',
                    'subsidies_2018' : 'Subsidies 2018',
                    'subsidies_2021' : 'Subsidies 2021',
                    'subsidies_2024' : 'Subsidies 2024',
                    'zero_interest_loan' : 'Zero interest loan'}
    dict_output = {scenario_rename.get(k, k): v for k, v in dict_output.items()}

    order_scenarios = ['Carbon tax', 'Carbon tax + EU-ETS 2', 'Subsidies 2018', 'Subsidies 2021', 'Subsidies 2024', 'WCO 2018', 'WCO 2021', 'WCO 2024', 'Zero interest loan', 'Mandatory renovation', 'Ban gas']    
    order_scenarios = [i for i in order_scenarios if i in dict_output.keys()]


In [ ]:
# read results
if False:
    dict_output = {f.split('_')[-1].split('.')[0] : pd.read_csv(os.path.join(folder, f), header=[0], index_col=[0]) for f in os.listdir('output') if f.endswith('.csv')}
    # transform column names to int
    for scenario in dict_output.keys():
        dict_output[scenario].columns = [int(c) for c in dict_output[scenario].columns]

    # replace centralized key with 'reference'
    dict_output['Reference'] = dict_output.pop('centralized')

    # add 2050 columns that is equal to 2049 in all scenarios
    for scenario in dict_output.keys():
        dict_output[scenario][2050] = dict_output[scenario][2049]
        

In [ ]:
# reference, order_scenarios = 'NoPolicy', None
config_policies = get_json('project/input/policies/cba_inputs.json')

config_policies['Discount rate'] = 0.032
config_policies['Lifetime'] = 20
config_policies['Factor COFP'] = 0.2
#config_policies['energy_prices'] = 'project/input/energy/energy_prices_wt_constant.csv'
#_ = indicator_policies(dict_output, folder, config_policies, reference=reference, order_scenarios=order_scenarios)


In [ ]:
if True:
    try:
        # add indicator policies in result
        indicator = pd.read_csv(os.path.join(folder, 'policies', 'indicator.csv'), header=[0], index_col=[0])
        for scenario in dict_output.keys():
            temp = 0
            if scenario in indicator.columns:
                temp = indicator.loc['NPV (Billion euro)', scenario]
            dict_output[scenario].loc['NPV (Billion Euro)', dict_output[scenario].columns[-1]] = temp
    except FileNotFoundError:
        pass
    
    # reference, scenario_assessment, order_scenarios, colors_scenarios = 'Reference', None, None, None
    scenario_assessment = [i for i in dict_output.keys() if i != reference]
    plot_compare_scenarios(deepcopy(dict_output), folder, quintiles=True, order_scenarios=order_scenarios, reference=reference, colors=colors_scenarios,
                           scenario_assessment=scenario_assessment)
    make_summary(folder)
